# E10 — O desenho da intervenção

O capítulo anterior terminou com duas famílias de explicação empatadas no mesmo dado: uma diz que
a oscilação de amanhã depende do que a série fez hoje, outra que existe um estado escondido que
troca de lei sem aparecer. Acumular mundo observado não escolheu entre elas. A tentativa concreta
que este caderno mede é a única saída que sobra: **segurar o mundo parado por alguns dias e olhar
como ele volta** — e, principalmente, **quanto isso custa em dado que não existe**.

O ato é o mesmo nos dois mundos: a série vai a zero por decreto durante a contenção, e nada mais
muda. O que muda é o que ele significa. Num mundo em que a memória mora na própria série, parar
apaga a memória acumulada, e o mundo volta ferido — mais ferido quanto mais tempo ficou parado.
Num mundo de estado escondido, o estado continua correndo enquanto a série está parada, e a
soltura devolve o mundo no nível de antes. Uma curva que sobe contra uma reta.

A medida é a função intervencao.resposta: o nível médio depois da soltura **contra o nível médio
do próprio mundo** na janela que antecede a contenção. Medir contra a escala de fora esconderia
justamente o que se quer ver.


In [1]:
# <- brinque com: SEMENTES, TOLERANCIA, CONTENCOES, ESPERA, REPLICATAS, DIFERENCAS, META
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, intervencao, regimes, volatilidade

SEMENTES = 12
CHAVES = ("taxa", "pior", "mediana", "acima_do_dobro")
TOLERANCIA = {"taxa": 0.002, "pior": 2.0, "mediana": 1.0, "acima_do_dobro": 0.03}
CONTENCOES = (1, 2, 5, 10, 20, 40, 80, 160, 252)
ESPERA = 20
REPLICATAS = 60
DIFERENCAS = (0.15, 0.20, 0.30)
META = 40
SEMENTE = 2024

serie = volatilidade.retornos_log(dados.carregar_serie("sp500.csv")).to_numpy()
SIGMA = float(serie.std(ddof=1))
PAR_MEMORIA = {"alfa": intervencao.ALFA, "beta": intervencao.BETA}
PAR_ESCONDIDO = {"p": intervencao.P_AGITADO, "razao": intervencao.RAZAO,
                 "permanencia": intervencao.PERMANENCIA}

empate = {"memoria": intervencao.leituras("memoria", SIGMA, len(serie), SEMENTES, **PAR_MEMORIA),
          "escondido": intervencao.leituras("escondido", SIGMA, len(serie), SEMENTES,
                                            **PAR_ESCONDIDO)}
margens = {c: empate["memoria"][c] - empate["escondido"][c] for c in CHAVES}
print("serie: %d dias | sigma %.5f | sementes %d" % (len(serie), SIGMA, SEMENTES))
print()
linhas = []
for c in CHAVES:
    linhas.append({"leitura": c, "memoria": empate["memoria"][c], "escondido": empate["escondido"][c],
                   "diferenca": margens[c], "tolerancia": TOLERANCIA[c],
                   "cabe": abs(margens[c]) <= TOLERANCIA[c]})
tabela_empate = pd.DataFrame(linhas).set_index("leitura")
print(tabela_empate.round(4).to_string())
print()
print("os dois empatam nas quatro: %s" % bool(tabela_empate["cabe"].all()))


serie: 6718 dias | sigma 0.01213 | sementes 12

                memoria  escondido  diferenca  tolerancia  cabe
leitura                                                        
taxa             0.0513     0.0521    -0.0008       0.002  True
pior            11.0000    13.0000    -2.0000       2.000  True
mediana          3.0000     3.0000     0.0000       1.000  True
acima_do_dobro   0.0627     0.0448     0.0179       0.030  True

os dois empatam nas quatro: True


## O que o empate mostra

As quatro leituras do capítulo anterior — a taxa de rompimento, o pior bloco de sessenta dias, a
mediana e o excesso de blocos cheios — não separam os dois mundos na tolerância declarada. O
empate é apertado onde sempre foi: **o pior bloco**, que é a estatística que também era o gargalo
quando se procurou uma lei compartilhada entre mercados. Observar mais um mundo da mesma família
não desfaria o empate, porque os dois mundos já passam pelo mesmo crivo.

É contra esse par que a intervenção é medida, e a pergunta que o resto do caderno responde é de
preço: quantos dias de experimento custa o que a observação não deu.


In [2]:
# O desenho: a resposta contra o tempo de contencao, e o preco de separar.
desenho = intervencao.desenho(np.random.default_rng(SEMENTE), SIGMA, CONTENCOES, ESPERA,
                              REPLICATAS, DIFERENCAS)
tabela = pd.DataFrame({
    "memoria": desenho["memoria_media"], "memoria_dp": desenho["memoria_desvio"],
    "escondido": desenho["escondido_media"], "escondido_dp": desenho["escondido_desvio"],
    "diferenca": desenho["diferenca_medida"], "desvio_somado": desenho["desvio_somado"],
    "replicatas": desenho["aperta_replicatas"], "dias": desenho["aperta_dias"]},
    index=pd.Index(desenho["contencoes"], name="dias de contenção"))
print(tabela.round(3).to_string())
print()
for d in DIFERENCAS:
    if d in desenho["melhor"]:
        m = desenho["melhor"][d]
        print("declarando %.2f: k=%d, %.1f replicatas, %.0f dias de experimento"
              % (d, m["contencao"], m["replicatas"], m["dias"]))
    else:
        print("declarando %.2f: nenhuma contencao entrega essa diferenca" % d)
print()
a = desenho["apertado"]
print("o desenho apertado (declara o que a contencao entrega): k=%d, diferenca %.3f, "
      "%.1f replicatas, %.0f dias" % (a["contencao"], a["diferenca"], a["replicatas"], a["dias"]))


                   memoria  memoria_dp  escondido  escondido_dp  diferenca  desvio_somado  replicatas      dias
dias de contenção                                                                                              
1                    0.980       0.298      1.058         0.293     -0.077          0.591     224.184  4707.860
2                    1.024       0.224      0.980         0.229      0.044          0.452     407.104  8956.281
5                    0.941       0.236      1.012         0.234     -0.071          0.470     167.338  4183.442
10                   0.928       0.241      1.001         0.270     -0.073          0.512     189.882  5696.473
20                   0.875       0.197      1.009         0.223     -0.134          0.420      37.594  1503.761
40                   0.802       0.233      1.008         0.282     -0.206          0.514      23.989  1439.316
80                   0.701       0.195      1.026         0.265     -0.324          0.460       7.729   

## O que a tabela diz

A resposta da memória cai com o tempo de contenção e a do estado escondido fica parada: é a
diferença de forma que a observação não via. Mas a curva tem duas pontas, e as duas custam.

Contenção curta não produz diferença suficiente — a coluna da diferença mostra que segurar o mundo
por um dia devolve quase o mesmo dos dois lados —, e contenção longa separa mais mas paga duas
vezes: cada replicata dura mais dias, e a dispersão da resposta no mundo escondido **cresce** com
o tempo de contenção. O motivo é o próprio estado escondido: quanto mais longa a contenção, maior
a chance de ela atravessar um episódio agitado, e aí a resposta deixa de ser comparável entre
replicatas. O ótimo é interior por isso, e não por gosto.

Nada disso é desenho de mercado: intervenção assim não existe para um índice. É desenho de
sistema — uma rede, uma máquina, uma planta que se pode segurar.


In [3]:
# A promessa do desenho, medida: com esse orcamento, quantas vezes separa de fato.
apertado = desenho["apertado"]
orcamento = apertado["dias"]
mesma_conta_curta = int(np.ceil(orcamento / (CONTENCOES[1] + ESPERA)))
SEMENTE_SEPARA = 11
curta = intervencao.separa(np.random.default_rng(SEMENTE_SEPARA), SIGMA, CONTENCOES[1], ESPERA,
                           mesma_conta_curta, meta=META)
poucas = intervencao.separa(np.random.default_rng(SEMENTE_SEPARA), SIGMA, apertado["contencao"],
                            ESPERA, 2, meta=META)
justa = intervencao.separa(np.random.default_rng(SEMENTE_SEPARA), SIGMA, apertado["contencao"],
                           ESPERA, apertado["replicatas"], meta=META)
dobro = intervencao.separa(np.random.default_rng(SEMENTE_SEPARA), SIGMA, apertado["contencao"],
                           ESPERA, 2 * apertado["replicatas"], meta=META)
quatro = intervencao.separa(np.random.default_rng(SEMENTE_SEPARA), SIGMA, apertado["contencao"],
                            ESPERA, 4 * apertado["replicatas"], meta=META)
print("mesmo orcamento (%d dias) gasto em contencao de %d dias: %.1f replicatas, separa em %.0f%%"
      % (orcamento, CONTENCOES[1], mesma_conta_curta, 100 * curta["fracao"]))
print("a contencao certa com duas replicatas: separa em %.0f%%" % (100 * poucas["fracao"]))
print("o desenho da conta, %.1f replicatas: separa em %.0f%% (a conta prometia %.0f%%)"
      % (justa["repeticoes"], 100 * justa["fracao"], 100 * justa["confianca"]))
print("o dobro, %.0f replicatas: %.0f%%" % (dobro["repeticoes"], 100 * dobro["fracao"]))
print("o quadruplo, %.0f replicatas: %.0f%%" % (quatro["repeticoes"], 100 * quatro["fracao"]))
entregue = justa
for tentativa in (justa, dobro, quatro):
    if tentativa["fracao"] >= tentativa["confianca"]:
        entregue = tentativa
        break
print()
print("o desenho que entrega a confianca declarada: %.0f replicatas de %d dias, %.0f dias de experimento"
      % (entregue["repeticoes"], apertado["contencao"] + ESPERA,
         entregue["repeticoes"] * (apertado["contencao"] + ESPERA)))


mesmo orcamento (772 dias) gasto em contencao de 2 dias: 36.0 replicatas, separa em 0%
a contencao certa com duas replicatas: separa em 32%
o desenho da conta, 8.0 replicatas: separa em 45% (a conta prometia 95%)
o dobro, 16 replicatas: 80%
o quadruplo, 31 replicatas: 100%

o desenho que entrega a confianca declarada: 31 replicatas de 100 dias, 3100 dias de experimento


## O teste que fecha o capítulo

O orçamento, gasto na contenção errada, compra zero — e compra zero **medido**, não argumentado: a
mesma quantidade de dias, aplicada à contenção curta, separa os dois mundos em nenhuma das
repetições do experimento inteiro.

O desenho que a conta comprou não entrega o que promete. A conta supõe que a resposta se comporta
como média de muitos sorteios, e a resposta tem cauda gorda: o estado escondido às vezes atravessa
o experimento inteiro, e aí aquela replicata não é comparável com as outras. Comprar o dobro e o
quádruplo das replicatas mostra o preço honesto — e é esse número, e não o da conta, que vale.

Fica a regra que este capítulo assina: **o preço de um desenho é medido, não calculado.** A conta
diz onde procurar; a repetição diz quanto custa.


In [4]:
# Figura 1: a resposta contra o tempo de contencao, com a dispersao desenhada.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
raios = 1.959963985 / np.sqrt(REPLICATAS)
for nome, medias, desvios, cor, rotulo in (
        ("memoria", desenho["memoria_media"], desenho["memoria_desvio"], "#1f4e79",
         "a memória mora na série"),
        ("escondido", desenho["escondido_media"], desenho["escondido_desvio"], "#b03a2e",
         "o estado e escondido")):
    eixo.errorbar(desenho["contencoes"], medias, yerr=raios * np.array(desvios), marker="o",
                  capsize=3, color=cor, label=rotulo, lw=1.6)
eixo.axhline(1.0, color="#555555", ls=":", lw=1.2)
eixo.set_xscale("log")
eixo.set_xticks(desenho["contencoes"])
eixo.set_xticklabels([str(k) for k in desenho["contencoes"]], fontsize=9)
eixo.set_xlabel("dias segurando o mundo parado")
eixo.set_ylabel("nível depois da soltura / nível antes")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E10_desenho_da_intervencao", 1)
plt.close(fig)
print("fundo da memoria: %.3f em %d dias"
      % (min(desenho["memoria_media"]), desenho["contencoes"][int(np.argmin(desenho["memoria_media"]))]))


fundo da memoria: 0.616 em 252 dias


In [5]:
# Figura 2: o preco do desenho apertado, e a promessa do desenho medida por repeticao.
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.8, 4.2))
esq.plot(desenho["contencoes"], desenho["aperta_dias"], marker="o", color="#1f4e79", lw=1.6)
menor = int(np.argmin(desenho["aperta_dias"]))
esq.scatter([desenho["contencoes"][menor]], [desenho["aperta_dias"][menor]], s=140, marker="*",
            color="#b03a2e", zorder=5, label="o melhor desenho")
esq.set_xscale("log")
esq.set_xticks(desenho["contencoes"])
esq.set_xticklabels([str(k) for k in desenho["contencoes"]], fontsize=9)
esq.set_xlabel("dias segurando o mundo parado")
esq.set_ylabel("dias de experimento")
esq.legend(frameon=False, fontsize=9)
esq.grid(alpha=0.25)
postos = np.arange(4)
fracoes = [100 * curta["fracao"], 100 * poucas["fracao"], 100 * justa["fracao"],
           100 * entregue["fracao"]]
rotulos = ["contenção de %d dias,\nmesmo orçamento" % CONTENCOES[1],
           "contenção de %d dias,\nduas replicatas" % apertado["contencao"],
           "contenção de %d dias,\n%.0f replicatas" % (apertado["contencao"], justa["repeticoes"]),
           "contenção de %d dias,\n%.0f replicatas" % (apertado["contencao"], entregue["repeticoes"])]
dir_.bar(postos, fracoes, 0.6, color=["#b03a2e", "#7f8c8d", "#1f4e79", "#2e7d32"])
for i, f in enumerate(fracoes):
    dir_.annotate("%.0f%%" % f, (i, f), textcoords="offset points", xytext=(0, 4), ha="center",
                  fontsize=9)
dir_.axhline(100 * justa["confianca"], color="#555555", ls="--", lw=1.2,
             label="a confianca declarada")
dir_.set_xticks(postos)
dir_.set_xticklabels(rotulos, fontsize=8)
dir_.set_ylabel("repetições em que separa (%)")
dir_.legend(frameon=False, fontsize=8)
dir_.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E10_desenho_da_intervencao", 2)
plt.close(fig)
print("fracoes que separam: %s" % [round(f, 1) for f in fracoes])


fracoes que separam: [0.0, 32.5, 45.0, 100.0]


## Leitura visual das figuras

Feita abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

Figura 1 (a resposta contra o tempo de contenção). O eixo horizontal é logarítmico e traz nove contenções, de um a duzentos e cinquenta e dois dias; o vertical é a razão entre o nível depois da soltura e o nível antes, de 0,6 a 1,1, com a linha pontilhada em 1. As duas curvas começam praticamente juntas e se separam depressa: a azul, a memória que mora na série, sobe um pouco no segundo dia e depois desce sem parar até o fundo do quadro; a vermelha, o estado escondido, volta ao um no segundo dia e passa todo o resto do gráfico oscilando em torno da linha pontilhada, sem tendência. Cada ponto traz a sua barra de erro, e nas contenções curtas os dois intervalos se sobrepõem: em um, dois e cinco dias não há vão entre as barras, e a contenção curta não separa nada a olho nu. O vão aparece por volta de vinte dias e é inequívoco nos dois pontos da direita, onde os intervalos não se tocam. Duas coisas enganam. O logaritmo dá o mesmo espaço ao salto de um para dois dias e ao trecho de oitenta a duzentos e cinquenta e dois, de modo que a parte do gráfico que decide o experimento é a que está espremida na direita. E a curva vermelha parece estável por andar colada na linha, quando a maior barra de erro do quadro é justamente a dela no extremo direito: a quietude é do traço, não do intervalo.

Figura 2 (o preço do desenho, nos dois painéis). No painel da esquerda o custo em dias de experimento não é uma tigela. A curva sobe do primeiro dia para o segundo, onde bate no topo do quadro, desce no quinto, volta a subir no décimo e só então cai, em passos irregulares, até o mínimo em oitenta dias, o ponto marcado com a estrela vermelha, subindo outra vez em cento e sessenta e duzentos e cinquenta e dois. O serrilhado é a informação: o custo não é função suave da contenção, ele vem do número inteiro de replicatas que cada diferença medida exige. O eixo vertical vai até oito mil e é esticado por aquele pico de dois dias, o que achata todo o resto, e o eixo horizontal, logarítmico outra vez, encosta o melhor desenho quase no extremo direito, embora oitenta dias sejam menos de um terço dos duzentos e cinquenta e dois. No painel da direita, quatro barras na mesma régua, com a linha tracejada da confiança declarada perto do topo. A primeira barra não tem altura nenhuma: só o rótulo flutua sobre o eixo, e ali está o orçamento inteiro gasto na contenção errada. A segunda, em cinza, sobe menos de um terço; a terceira, que é o desenho que a conta comprou, fica visivelmente abaixo da linha tracejada; e só a quarta, em verde, encosta no topo. A distância entre a terceira e a linha é o preço que a conta não cobrou, e é a única coisa que essa figura precisa mostrar.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "desenho_dias_serie": int(len(serie)),
    "desenho_sementes_empate": int(SEMENTES),
    "desenho_replicatas_por_celula": int(REPLICATAS),
    "desenho_dias_por_replicata_maior": int(CONTENCOES[-1] + ESPERA),
    "desenho_espera": int(ESPERA),
    "desenho_meta": int(META),
    "desenho_empate_pior_diferenca": float(margens["pior"]),
    "desenho_empate_tolerancia_pior": float(TOLERANCIA["pior"]),
    "desenho_resposta_memoria_descida": float(desenho["memoria_media"][0] - min(desenho["memoria_media"])),
    "desenho_diferenca_maior": float(max(abs(np.array(desenho["diferenca_medida"])))),
    "desenho_diferenca_maior_contensao": int(desenho["contencoes"][int(np.argmax(np.abs(desenho["diferenca_medida"])))]),
    "desenho_desvio_menor": float(min(desenho["desvio_somado"])),
    "desenho_desvio_maior": float(max(desenho["desvio_somado"])),
    "desenho_apertado_contensao": int(apertado["contencao"]),
    "desenho_apertado_diferenca": float(apertado["diferenca"]),
    "desenho_apertado_replicatas": float(apertado["replicatas"]),
    "desenho_apertado_dias": float(apertado["dias"]),
    "desenho_apertado_dias_curta": float(desenho["aperta_dias"][0]),
    "desenho_mesma_conta_replicatas": int(mesma_conta_curta),
    "desenho_separa_curta_pct": float(100 * curta["fracao"]),
    "desenho_separa_poucas_pct": float(100 * poucas["fracao"]),
    "desenho_separa_planejada_pct": float(100 * justa["fracao"]),
    "desenho_separa_dobro_pct": float(100 * dobro["fracao"]),
    "desenho_separa_quadruplo_pct": float(100 * quatro["fracao"]),
    "desenho_entregue_replicatas": float(entregue["repeticoes"]),
    "desenho_entregue_dias": float(entregue["repeticoes"] * (apertado["contencao"] + ESPERA)),
    "desenho_entregue_pct": float(100 * entregue["fracao"]),
    "desenho_confianca_pct": float(100 * justa["confianca"]),
}
NOMES_CONTENCAO = {1: "um", 2: "dois", 5: "cinco", 10: "dez", 20: "vinte", 40: "quarenta",
                  80: "oitenta", 160: "cento_e_sessenta", 252: "duzentos_e_cinquenta_e_dois"}
for i, k in enumerate(desenho["contencoes"]):
    nome = NOMES_CONTENCAO[k]
    resultado["desenho_resposta_memoria_%s" % nome] = float(desenho["memoria_media"][i])
    resultado["desenho_resposta_escondido_%s" % nome] = float(desenho["escondido_media"][i])
    resultado["desenho_diferenca_%s" % nome] = float(desenho["diferenca_medida"][i])
    resultado["desenho_custo_%s" % nome] = float(desenho["aperta_dias"][i])

for d in DIFERENCAS:
    nome = {0.15: "quinze_centesimos", 0.20: "vinte_centesimos", 0.30: "trinta_centesimos"}[d]
    if d in desenho["melhor"]:
        resultado["desenho_%s_contensao" % nome] = int(desenho["melhor"][d]["contencao"])
        resultado["desenho_%s_replicatas" % nome] = float(desenho["melhor"][d]["replicatas"])
        resultado["desenho_%s_dias" % nome] = float(desenho["melhor"][d]["dias"])

caminho = Path("lab/resultados/E10_desenho_da_intervencao.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True),
                   encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E10_desenho_da_intervencao.json gravado | 73 grandezas
